# Error Handling in Practice

## What you'll learn

- How the framework catches and records runtime failures
- Finding and reading failure logs
- Partial failures with `FailurePolicy` (CONTINUE vs FAIL_FAST)
- Empty input cascades from strict filters
- Validation errors vs runtime errors

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb)  
**Estimated time:** 15 minutes  
**GPU required:** No.

---

Inspect the outcomes of invalid data, partial failures, filters that reject
every artifact, and invalid configuration.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter, IngestCSV
from artisan.operations.examples import DataGeneratorWithMetrics, DataTransformer
from artisan.orchestration import PipelineManager, StepStatus
from artisan.schemas import FailurePolicy
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_failures, inspect_pipeline

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

## A step that fails

Create a CSV with `"INVALID"` in a numeric column, ingest it, then pass it to
`DataTransformer`. `IngestCSV` reads the CSV headers and row count without
validating numeric values. The transformer fails when it tries `float("INVALID")`.

In [ ]:
import os

env = tutorial_setup("error_handling")

# Create a CSV with a non-numeric value in the 'score' column
bad_csv = os.path.join(env.working_root, "bad_data.csv")
with open(bad_csv, "w") as f:
    f.write("id,x,y,z,score\n0,1.0,2.0,3.0,INVALID\n")
print(f"Bad CSV: {bad_csv}")
with open(bad_csv) as f:
    print(f.read())

In [ ]:
pipeline = PipelineManager.create(
    name="error_handling",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: Ingest the CSV (succeeds without validating numeric values)
step_0 = pipeline.run(
    definition=IngestCSV,
    name="ingest",
    inputs=[bad_csv],
)

# Step 1: Transform — fails when parsing the non-numeric value
step_1 = pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)

result = pipeline.finalize()
assert step_0.status is StepStatus.SUCCEEDED
assert step_1.status is StepStatus.FAILED
print(f"Pipeline complete: success={result['overall_success']}")

In [ ]:
print(
    f"Step 0 (ingest):    status={step_0.status.value}, succeeded={step_0.succeeded_count}, failed={step_0.failed_count}"
)
print(
    f"Step 1 (transform): status={step_1.status.value}, succeeded={step_1.succeeded_count}, failed={step_1.failed_count}"
)

In [ ]:
inspect_pipeline(env.delta_root)

`inspect_pipeline` shows `"failed"` because no transform execution succeeded.
Under `CONTINUE`, a mixture of successes and failures would be `"partial"`.

Use `inspect_failures` below to read execution errors, including `code`,
`recovery_hint`, and `field` when the failure is structured.

## Finding and reading failure logs

Optional execution diagnostics include a bounded, redacted failure log when the
worker can publish it. Use the inspection API to select the exact failed
execution and resolve its recorded log location. Missing optional logs remain
unavailable rather than changing the durable failure result.
See [Debug Executions](../../how-to-guides/debugging-executions.md) for replay
configuration and secret handling.

In [ ]:
from artisan.inspection import inspect_evidence

excerpt = inspect_evidence(
    "get_step_logs",
    {
        "pipeline_run_id": pipeline.config.pipeline_run_id,
        "step_name": "transform",
        "tail_lines": 50,
    },
    delta_root=env.delta_root,
)
assert excerpt.lines
print("\n".join(excerpt.lines))

The log shows the conversion error and the traceback pointing to
`data_transformer.py`. Its execution ID identifies the selected outer execution;
use the inspection API to cross-reference its failure and command evidence.

In [ ]:
inspect_failures(env.delta_root)

## Partial failures with FailurePolicy

`FailurePolicy` determines what happens when some inputs fail and others succeed:

| Policy | Behavior |
|--------|----------|
| `CONTINUE` (default) | Commit successful results and return a `partial` StepResult |
| `FAIL_FAST` | Return a `failed` StepResult; already-finished records remain available for audit |

We'll create 5 CSVs where 2 have invalid data, then run with each
policy.

In [ ]:
import os

env_partial = tutorial_setup("error_handling_partial")

# Create 5 CSVs — indices 1 and 3 have non-numeric values
csv_dir = os.path.join(env_partial.working_root, "csvs")
os.makedirs(csv_dir, exist_ok=True)

csv_paths = []
for i in range(5):
    path = os.path.join(csv_dir, f"data_{i:03d}.csv")
    if i in (1, 3):
        content = f"id,x,y,z,score\n{i},1.0,2.0,3.0,INVALID\n"
    else:
        content = f"id,x,y,z,score\n{i},1.0,2.0,3.0,0.5\n"
    with open(path, "w") as f:
        f.write(content)
    csv_paths.append(path)

print(f"Created {len(csv_paths)} CSVs (indices 1, 3 have bad data)")

In [ ]:
pipeline = PipelineManager.create(
    name="partial_continue",
    delta_root=env_partial.delta_root,
    staging_root=env_partial.staging_root,
    working_root=env_partial.working_root,
)
output = pipeline.output

step_0 = pipeline.run(
    definition=IngestCSV,
    name="ingest",
    inputs=csv_paths,
)

# Default policy is CONTINUE — failures are logged, successes committed
step_1 = pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)

result = pipeline.finalize()
assert step_1.status is StepStatus.PARTIAL
assert step_1.succeeded_count == 3
assert step_1.failed_count == 2

print(f"Step 1: succeeded={step_1.succeeded_count}, failed={step_1.failed_count}")
print(f"Pipeline success={result['overall_success']}")

In [ ]:
inspect_pipeline(env_partial.delta_root)

With `CONTINUE`, the step became `partial` with 3 successes and 2 failures.
The 3 successful outputs were committed to Delta Lake. Any downstream
step would receive only those 3 artifacts.

Run the same pipeline with `FAIL_FAST`:

In [ ]:
env_fast = tutorial_setup("error_handling_failfast")

pipeline = PipelineManager.create(
    name="partial_failfast",
    delta_root=env_fast.delta_root,
    staging_root=env_fast.staging_root,
    working_root=env_fast.working_root,
    failure_policy=FailurePolicy.FAIL_FAST,
)
output = pipeline.output

step_0 = pipeline.run(
    definition=IngestCSV,
    name="ingest",
    inputs=[str(p) for p in csv_paths],
)

step_1 = pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)
assert step_1.status is StepStatus.FAILED
print(f"FAIL_FAST step status: {step_1.status.value}")
pipeline.finalize()

With `FAIL_FAST`, the step is `failed` and exposes no output roles to
downstream steps. Failure records and sibling work that already finished
remain persisted for audit. Use `FAIL_FAST` when partial output is not meaningful.

## Empty input cascades

When a filter rejects every artifact, the pipeline skips downstream work and
records why. Generate `mean_score` values in [0, 1], then filter with the
impossible threshold > 100.

In [ ]:
env_cascade = tutorial_setup("error_handling_cascade")

pipeline = PipelineManager.create(
    name="empty_cascade",
    delta_root=env_cascade.delta_root,
    staging_root=env_cascade.staging_root,
    working_root=env_cascade.working_root,
)
output = pipeline.output

step_0 = pipeline.run(
    definition=DataGeneratorWithMetrics,
    name="generate",
    params={"count": 5, "seed": 42},
)

step_1 = pipeline.run(
    definition=Filter,
    name="filter",
    inputs={"passthrough": output("generate", "datasets")},
    params={
        "criteria": [{"metric": "mean_score", "operator": "gt", "value": 100}],
    },
)

step_2 = pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("filter", "passthrough")},
)

result = pipeline.finalize()
assert step_2.status is StepStatus.SKIPPED
assert step_2.metadata["skip_reason"] == "empty_inputs"
print(f"Pipeline complete: success={result['overall_success']}")

In [ ]:
inspect_pipeline(env_cascade.delta_root)

In [ ]:
for step in [step_0, step_1, step_2]:
    if step.status is StepStatus.SKIPPED:
        reason = step.metadata["skip_reason"]
        print(f"Step {step.step_number} ({step.step_name}): skipped — {reason}")
    else:
        print(
            f"Step {step.step_number} ({step.step_name}): {step.succeeded_count} succeeded"
        )

The filter passed zero artifacts, so step 2 was skipped with
`skip_reason="empty_inputs"`. If there were more downstream steps,
they would show `"pipeline_stopped"`.

:::{note}
To evaluate filter criteria without removing artifacts, use
`passthrough_failures=True` in the Filter params. This records
which artifacts would fail but passes everything through.
:::

## Validation before execution

Parameter names and values are validated at submission before a step attempt is
created. Invalid values raise `ValidationError`; no worker is dispatched for
that rejected submission. You can also validate values independently by
constructing the operation's `Params` model.

In [ ]:
import pydantic

validation_errors = []
try:
    DataTransformer.Params(scale_factor="not_a_number")
except pydantic.ValidationError as exc:
    validation_errors = exc.errors()
assert validation_errors[0]["loc"] == ("scale_factor",)

env_val = tutorial_setup("error_handling_validation")
with PipelineManager.create(
    name="validation_demo",
    delta_root=env_val.delta_root,
    staging_root=env_val.staging_root,
    working_root=env_val.working_root,
) as pipeline:
    source = pipeline.run(DataGeneratorWithMetrics, params={"count": 1, "seed": 42})
    submission_errors = []
    try:
        pipeline.run(
            DataTransformer,
            name="transform",
            inputs={"dataset": source.output("datasets")},
            params={"scale_factor": "not_a_number"},
        )
    except pydantic.ValidationError as exc:
        submission_errors = exc.errors()
    assert submission_errors[0]["loc"] == ("scale_factor",)
    assert pipeline.finalize()["overall_success"]

attempts = inspect_pipeline(env_val.delta_root)
assert attempts.height == 1, "Rejected parameters must not allocate a step attempt"
attempts

Both explicit construction and submission rejected the invalid parameter.
Only the valid source step appears in the run; the rejected transform has no
worker execution or failure log. The bad-CSV example above reached a worker,
so its error is available through `inspect_failures` and optional log evidence.

## Next steps

- [Storage Layout and Logging](../06-storage/01-storage-and-logging.ipynb) —
  Directory layout, Delta Lake tables, and logging configuration
- [Error Handling](../../concepts/error-handling.md) — Design
  rationale for failure containment
- [Resume and Caching](../03-caching/01-resume-and-caching.ipynb) — How caching
  interacts with failures during re-runs